# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Random Forest classifier with class_weight='balanced', chosen for this lane over a linear model because last week's plain logistic regression on the same five features produced near-zero recall on the imbalanced decline label (roughly 67/33). A tree ensemble can capture non-linear thresholds and interactions between position, click volume, and traffic mix that a linear model can't, and the balanced class weighting directly addresses the label imbalance rather than letting the majority class dominate.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Grouped by client_hash_id using GroupShuffleSplit, not a plain random row split. Content items from the same client share underlying account-level factors (niche, GSC connection quality, traffic patterns), so a random split would let the model see other content from the same client in training and test at once, an easy way to overstate performance. Verified zero client overlap between train and test before trusting any metric. Not time-aware beyond what's already baked into the label: features come from January-February, the label from March, so the split respects that ordering by construction, it's purely a client-grouped split within that fixed feature/label boundary.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same test rows, same features available at decision time, same metrics computed both ways.

method	auc	precision	recall
w04_baseline_ctr_gap	0.165	0.053	0.050
w05_random_forest	0.872	0.620	0.973

The w04 baseline scored below 0.5 AUC, worse than random, meaning ctr_gap runs in the wrong direction as a decline predictor: it was built to flag CTR underperformance relative to position, not to predict future decline, and this comparison shows directly that repurposing it that way doesn't work. The Random Forest is a substantial, honest improvement on the same split and metric.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Permutation importance shows avg_daily_clicks dominating (0.254), with avg_daily_impressions next (0.042) and the remaining three features contributing very little. This needs a direct caveat: is_declining is defined as March average clicks falling below 80% of avg_daily_clicks, so this feature partly encodes the label's own reference point, not purely independent signal. The model may be leaning on that mechanical relationship more than on genuinely predictive behavior.

Error profile: 513 false positives against only 23 false negatives, so recall (0.973) is bought at the cost of precision (0.620), the model predicts decline liberally. The 23 false negatives it does miss share a clear profile: low volume (mean 0.29 avg daily clicks, mean 90 impressions) with predicted probabilities clustered tightly at 0.38-0.50, just under the decision threshold, genuinely borderline cases rather than confident misses.

Self-check: tick each box only once true, then run top to bottom, commit w05_model.ipynb under work/notebooks/, submit the repo URL.